In [33]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path("../../../../SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
    Path("backend/SOURCES_AND_DATASHEETS/usgs_data_USGS-01646500.csv"),
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate usgs_data_USGS-01646500.csv")
print(f"Using CSV file at: {csv_path}")

df = pd.read_csv(csv_path)
df

Using CSV file at: ..\..\..\..\SOURCES_AND_DATASHEETS\usgs_data_USGS-01646500.csv


,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,snowfall,snow_depth,soil_moisture_0_to_1cm,soil_moisture_1_to_3cm,temperature_2m,wind_speed_10m,vapour_pressure_deficit,precip_3hr,precip_24hr,precip_72hr
0,2010-07-06 00:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-07-06 00:15:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-07-06 00:30:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-07-06 00:45:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-07-06 01:00:00+00:00,2.73,1600.0,NaN,NaN,366.0,29.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
626719,2026-07-06 23:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,1.0,1.0,0.0,0.0,NaN,NaN,24.65,0.648999,0.118415,4.9,29.8,33.3
626720,2026-07-07 00:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,1.8,1.8,0.0,0.0,NaN,NaN,24.10,11.074022,0.062441,5.2,31.6,35.0
626721,2026-07-07 01:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,0.7,0.7,0.0,0.0,NaN,NaN,23.45,11.225132,0.051792,3.5,32.3,35.7
626722,2026-07-07 02:00:00+00:00,NaN,NaN,6.6,8.8,348.0,31.7,7.0,1.1,1.1,0.0,0.0,NaN,NaN,22.85,10.137692,0.016814,3.6,33.4,36.8


In [34]:
# Flood Action Stage: 5 ft
# Minor Flood Stage: 10 ft
# Moderate Flood Stage: 12 ft
# Major Flood Stage: 14 ft
FLOOD_ACTION_STAGE = 5.0
MINOR_FLOOD_STAGE = 10.0
MODERATE_FLOOD_STAGE = 12.0
MAJOR_FLOOD_STAGE = 14.0

#df.hist(figsize=(10, 6))
# get the instances where Gage Height is > 5
df = df.dropna(subset=['gage_height_ft'])
df_flood = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE]
df_minor_flood = df[df['gage_height_ft'] > MINOR_FLOOD_STAGE]
df_moderate_flood = df[df['gage_height_ft'] > MODERATE_FLOOD_STAGE]
df_major_flood = df[df['gage_height_ft'] > MAJOR_FLOOD_STAGE]

# print the lengths of each of the dataframes
print(f"Total records: {len(df)}")
print(f"Flood Action Stage records: {len(df_flood)}")
print(f"Minor flood records: {len(df_minor_flood)}")
print(f"Moderate flood records: {len(df_moderate_flood)}")
print(f"Major flood records: {len(df_major_flood)}")

Total records: 624490
Flood Action Stage records: 66623
Minor flood records: 1199
Moderate flood records: 55
Major flood records: 0


In [35]:

# Name the 'Unnamed: 0' column as 'datetime' and convert it to datetime type
df['datetime'] = pd.to_datetime(df['Unnamed: 0'])

df = df.sort_values('datetime').reset_index(drop=True)

# how many hours of gap counts as "the storm ended" (tune this to your data's
# sampling frequency, e.g. 6-12h for hourly gauge data, 24-48h for daily)
GAP_HOURS = 12

# isolate just the flagged (action-stage) rows
flood_rows = df[df['gage_height_ft'] > FLOOD_ACTION_STAGE].copy()

# time since previous flagged reading, saved in column 'gap'
flood_rows['gap'] = flood_rows['datetime'].diff()

# start a new event whenever the gap exceeds threshold (or it's the first row)
flood_rows['new_event'] = (
    flood_rows['gap'].isna() | (flood_rows['gap'] > pd.Timedelta(hours=GAP_HOURS))
)
flood_rows['event_id'] = flood_rows['new_event'].cumsum()

df = df.merge(
    flood_rows[['datetime', 'event_id']],
    on='datetime',
    how='left'
)

# summarize each event
events = flood_rows.groupby('event_id').agg(
    start=('datetime', 'min'),
    end=('datetime', 'max'),
    n_readings=('datetime', 'count'),
    peak_gage_height=('gage_height_ft', 'max')  # adjust column name as needed
).reset_index(drop=True)

events['duration_hours'] = (events['end'] - events['start']).dt.total_seconds() / 3600

print(f"Total flagged readings: {len(flood_rows)}")
print(f"Independent storm events: {len(events)}")
print(events)


C:\Users\drpri\AppData\Local\Temp\ipykernel_41520\2735280843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['datetime'] = pd.to_datetime(df['Unnamed: 0'])


Total flagged readings: 66623
Independent storm events: 123
                        start                       end  n_readings  \
0   2010-12-03 00:00:00+00:00 2010-12-05 12:15:00+00:00         242   
1   2011-02-27 11:30:00+00:00 2011-03-04 14:45:00+00:00         494   
2   2011-03-07 01:30:00+00:00 2011-03-19 02:30:00+00:00        1153   
3   2011-03-26 06:30:00+00:00 2011-03-28 07:30:00+00:00         194   
4   2011-04-13 08:45:00+00:00 2011-05-08 04:15:00+00:00        2382   
..                        ...                       ...         ...   
118 2025-06-10 16:00:00+00:00 2025-06-13 06:00:00+00:00         193   
119 2025-06-16 22:15:00+00:00 2025-06-22 17:45:00+00:00         544   
120 2025-07-19 23:50:00+00:00 2025-07-20 01:35:00+00:00          22   
121 2026-02-21 15:35:00+00:00 2026-02-24 23:50:00+00:00         964   
122 2026-05-24 19:00:00+00:00 2026-05-31 20:35:00+00:00        2036   

     peak_gage_height  duration_hours  
0                6.70       60.250000  
1      

In [36]:
import numpy as np

LOOKAHEAD_HOURS = 24  # predict within next 24h
FREQ_MINUTES = 15     # nominal sampling interval; cell 4 uses it as a staleness tolerance

# Every timestamp at which the river sits above action stage.
exceedance_times = df.loc[df['gage_height_ft'] > FLOOD_ACTION_STAGE, 'datetime'].values

# Window bounds per row: strictly after now, through now + 24h.
# Matched on time, not row count -- sampling rate varies and the record has gaps.
window_start = np.searchsorted(exceedance_times, df['datetime'].values, side='right')
window_end = np.searchsorted(
    exceedance_times,
    (df['datetime'] + pd.Timedelta(hours=LOOKAHEAD_HOURS)).values,
    side='right',
)

# 1 if any exceedance falls inside that window. This is the model's target.
df['will_flood'] = (window_end > window_start).astype(int)

# Drop the last 24h: no observable future, so any label there would be invented.
df = df[
    df['datetime'] <= df['datetime'].max() - pd.Timedelta(hours=LOOKAHEAD_HOURS)
].reset_index(drop=True)


In [37]:
import numpy as np

# How fast the river is rising: stage now minus stage 1h and 6h ago.
# Matched on time, not row count -- spacing varies and the record has gaps.
# Same quantity the live path builds with _diff_by_elapsed_time().

_reading_times = df['datetime'].values
_gage_heights = df['gage_height_ft'].values

# How stale the reference reading may be before the window counts as unusable.
_stale_tolerance = np.timedelta64(FREQ_MINUTES, 'm')


def _gage_change_over(hours):
    """Stage now minus stage `hours` ago, matched on time rather than row count."""
    target_times = _reading_times - np.timedelta64(int(hours * 60), 'm')

    # Most recent reading at or before each target time.
    reference_idx = np.searchsorted(_reading_times, target_times, side='right') - 1

    usable = (reference_idx >= 0) & (
        (target_times - _reading_times[np.clip(reference_idx, 0, None)]) <= _stale_tolerance
    )

    change = np.full(len(_reading_times), np.nan)
    change[usable] = _gage_heights[usable] - _gage_heights[reference_idx[usable]]
    return change


# NaN at the start of the record and across gaps; cell 5's dropna removes those rows.
df['gage_height_roc_1h'] = _gage_change_over(1)
df['gage_height_roc_6h'] = _gage_change_over(6)

df['gage_height_now'] = df['gage_height_ft']
df['streamflow_now'] = df['streamflow_cfs']

# Precip and turbidity are right-skewed; log1p keeps the tails from dominating.
for col in ['precip_3hr', 'precip_24hr', 'precip_72hr', 'turbidity_fnu']:
    if col in df.columns:
        df[f'{col}_log'] = np.log1p(df[col].clip(lower=0))

feature_columns = [
    # hydraulic — current state + trend
    'gage_height_ft',
    'gage_height_roc_1h',
    'gage_height_roc_6h',

    # precip — log-transformed versions only (not the raw skewed ones)
    'precip_3hr_log',
    'precip_24hr_log',
    'precip_72hr_log',

    # weather
    'temperature_2m',
    'wind_speed_10m',
    'vapour_pressure_deficit',
    'rain',
    'snowfall',
    'snow_depth',

    # water quality
    'specific_conductance_us_cm',
    'temperature_c',
]


In [38]:
# tag every row (not just flood rows) with which event's "influence window" it falls in
# so the same storm doesn't appear in both train and test
events_sorted = events.sort_values('start').reset_index(drop=True)

# hold out the most recent ~20% of events as test
n_test_events = int(len(events_sorted) * 0.2)
test_events = events_sorted.iloc[-n_test_events:]
train_events = events_sorted.iloc[:-n_test_events]

test_start_cutoff = test_events['start'].min() - pd.Timedelta(days=3)  # buffer before first test storm

train_df = df[df['datetime'] < test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])
test_df  = df[df['datetime'] >= test_start_cutoff].dropna(subset=feature_columns + ['will_flood'])

test_df

,Unnamed: 0,gage_height_ft,streamflow_cfs,dissolved_oxygen_mg_l,pH,specific_conductance_us_cm,temperature_c,turbidity_fnu,precipitation,rain,...,event_id,will_flood,gage_height_roc_1h,gage_height_roc_6h,gage_height_now,streamflow_now,precip_3hr_log,precip_24hr_log,precip_72hr_log,turbidity_fnu_log
430325,2022-12-14 13:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,-0.01,-0.03,3.13,3520.0,0.000000,0.000000,0.262364,0.875469
430326,2022-12-14 13:45:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.000000,0.262364,0.875469
430327,2022-12-14 14:00:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.000000,0.182322,0.875469
430328,2022-12-14 14:15:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.03,3.13,3520.0,0.000000,0.000000,0.182322,0.875469
430329,2022-12-14 14:30:00+00:00,3.13,3520.0,14.7,8.7,380.0,5.6,1.4,0.0,0.0,...,NaN,0,0.00,-0.02,3.13,3520.0,0.000000,0.000000,0.182322,0.875469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
624197,2026-07-04 23:40:00+00:00,2.82,1980.0,12.3,9.0,347.0,34.6,11.1,0.0,0.0,...,NaN,0,0.00,-0.02,2.82,1980.0,0.955511,1.029619,1.193922,2.493205
624198,2026-07-04 23:45:00+00:00,2.83,2030.0,12.3,9.0,347.0,34.6,11.1,0.0,0.0,...,NaN,0,0.00,-0.01,2.83,2030.0,0.955511,1.029619,1.193922,2.493205
624199,2026-07-04 23:50:00+00:00,2.82,1980.0,12.3,9.0,347.0,34.6,11.1,0.0,0.0,...,NaN,0,-0.01,-0.02,2.82,1980.0,0.955511,1.029619,1.193922,2.493205
624200,2026-07-04 23:55:00+00:00,2.82,1980.0,12.3,9.0,347.0,34.6,11.1,0.0,0.0,...,NaN,0,-0.01,-0.02,2.82,1980.0,0.955511,1.029619,1.193922,2.493205


In [39]:
"""This will be where the XGBoost model is going to be created."""
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

X_train, y_train = train_df[feature_columns], train_df['will_flood']
X_test, y_test = test_df[feature_columns], test_df['will_flood']

model = Pipeline([
    ("xgb", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    ))
])

model.fit(X_train, y_train)
probs = model.predict_proba(X_test)[:, 1]


In [ ]:
# ── Evaluation: measure this like the warning system it is ────────────────────
#
# Row-level metrics (PR-AUC, precision, recall) describe the sampling rate more than
# the product. The positive rows are 24 storm events sampled ~118 times each, and
# event detection saturates -- every threshold from 0.04 to 0.32 catches 24/24.
#
# What this system promises: warn before the river crosses action stage, early enough
# to act, without alarming so often people stop looking. Three measurable things.
#
# PRIMARY METRIC: mean lead time across all events, with a missed event scored as
# zero, at a fixed alert-time budget. Misses are punished, earlier warning is
# rewarded, and holding the budget fixed stops lead time being bought with noise.
#
# The budget is TIME SPENT ALERTING, not a count of alarms. Counting alarms rewards a
# system that switches on and never switches off: gage height alone scores 45h of
# lead and only 23 alarms a year that way, because one alarm can last for days.

from sklearn.metrics import brier_score_loss

ALERT_TIME_BUDGET = 0.02       # alert at most 2% of watch time. Policy choice; hold it fixed.
EVENT_LOOKBACK_HOURS = 72      # how far before an onset to look for the warning
BOOTSTRAP_SAMPLES = 2000

# A warning only means anything while the river is still below stage.
_watching = (test_df['gage_height_ft'] <= FLOOD_ACTION_STAGE).values
watch = test_df.loc[_watching, ['datetime', 'gage_height_ft', 'will_flood']].reset_index(drop=True)
watch['prob'] = probs[_watching]

onsets = pd.DatetimeIndex(events.loc[events['start'] >= test_start_cutoff, 'start'].sort_values())
span_years = (test_df['datetime'].max() - test_df['datetime'].min()).total_seconds() / (365.25 * 24 * 3600)
_times = watch['datetime']


def _alert_duty_cycle(scores, threshold):
    """Fraction of watch time the alert is on."""
    return float((scores >= threshold).mean())


def _false_alarms_per_year(scores, threshold):
    """Alert episodes with no onset in the following 24h, per year."""
    firing = scores >= threshold
    starts = np.where(firing & ~np.r_[False, firing[:-1]])[0]
    false = sum(
        1 for i in starts
        if not ((onsets >= _times.iloc[i]) & (onsets <= _times.iloc[i] + pd.Timedelta(hours=24))).any()
    )
    return false / span_years


def _lead_times(scores, threshold):
    """Hours of unbroken warning right before each onset. 0.0 means no warning."""
    leads = []
    for onset in onsets:
        window = ((_times >= onset - pd.Timedelta(hours=EVENT_LOOKBACK_HOURS)) & (_times < onset)).values
        firing = scores[window] >= threshold
        if firing.size == 0 or not firing[-1]:
            leads.append(0.0)                     # silent when the water arrived
            continue
        quiet = np.where(~firing)[0]
        first = 0 if quiet.size == 0 else quiet[-1] + 1
        leads.append((onset - _times[window].iloc[first]).total_seconds() / 3600)
    return np.array(leads)


def _bursts_per_event(scores, threshold):
    """Separate on/off alert bursts in the 24h before each onset."""
    counts = []
    for onset in onsets:
        window = ((_times >= onset - pd.Timedelta(hours=24)) & (_times < onset)).values
        firing = scores[window] >= threshold
        counts.append(0 if firing.size == 0 else int(np.sum(firing & ~np.r_[False, firing[:-1]])))
    return np.array(counts)


def _threshold_for_budget(scores, budget):
    """Lowest threshold whose alert time still fits the budget."""
    low, high = float(scores.min()) - 1e-9, float(scores.max()) + 1e-9
    for _ in range(60):
        mid = (low + high) / 2
        if _alert_duty_cycle(scores, mid) > budget:
            low = mid
        else:
            high = mid
    return high


def _evaluate(name, scores):
    threshold = _threshold_for_budget(scores, ALERT_TIME_BUDGET)
    leads = _lead_times(scores, threshold)
    bursts = _bursts_per_event(scores, threshold)

    # 24 events is a small sample, so the mean gets a confidence interval.
    rng = np.random.default_rng(0)
    booted = rng.choice(leads, size=(BOOTSTRAP_SAMPLES, leads.size), replace=True).mean(axis=1)
    low, high = np.percentile(booted, [2.5, 97.5])

    print(f"\n{name}")
    print(f"  operating threshold        {threshold:>8.4f}")
    print(f"  MEAN LEAD TIME             {leads.mean():>8.2f} h   (95% CI {low:.2f} to {high:.2f})")
    print(f"  events warned              {int((leads > 0).sum()):>8}/{leads.size}")
    print(f"  lead p10 / p50 / p90       {np.percentile(leads, 10):>8.1f} /{np.percentile(leads, 50):>6.1f} /"
          f"{np.percentile(leads, 90):>6.1f} h")
    print(f"  worst event                {leads.min():>8.2f} h")
    print(f"  time spent alerting        {_alert_duty_cycle(scores, threshold) * 100:>8.2f} %  "
          f"({_alert_duty_cycle(scores, threshold) * 8766:.0f} h/yr)")
    print(f"  false alarm episodes/yr    {_false_alarms_per_year(scores, threshold):>8.1f}")
    print(f"  alert bursts per event     {np.median(bursts):>8.1f}   (max {bursts.max()})")
    return leads


print("=" * 70)
print(f"WARNING PERFORMANCE  --  {len(onsets)} events over {span_years:.2f} years")
print(f"every system below is tuned to alert the same {ALERT_TIME_BUDGET * 100:.0f}% of the time")
print("=" * 70)

model_leads = _evaluate("MODEL", watch['prob'].values)
gage_leads = _evaluate("BASELINE -- gage height alone, no model", watch['gage_height_ft'].values)

print(f"\n  the model buys {model_leads.mean() - gage_leads.mean():+.2f} h of extra warning "
      f"for the same alert time")

# The chapter page shows this probability to the public, so it should mean what it says.
print("\n" + "=" * 70)
print("CALIBRATION  --  do the probabilities we display mean anything?")
print("=" * 70)
observed = watch['will_flood'].values
brier = brier_score_loss(observed, watch['prob'])
climatology = brier_score_loss(observed, np.full(observed.size, observed.mean()))
print(f"  Brier {brier:.5f}   vs base-rate guess {climatology:.5f}   skill {1 - brier / climatology:+.3f}")
print(f"\n  {'band':>14}{'n':>10}{'predicted':>12}{'observed':>11}")
for low_edge, high_edge in [(0, .05), (.05, .1), (.1, .2), (.2, .4), (.4, .6), (.6, .8), (.8, 1.01)]:
    band = ((watch['prob'] >= low_edge) & (watch['prob'] < high_edge)).values
    if band.sum() == 0:
        continue
    label = f"{low_edge:.2f}-{high_edge:.2f}"
    print(f"  {label:>14}{int(band.sum()):>10,}{watch.loc[band, 'prob'].mean():>12.3f}"
          f"{observed[band].mean():>11.3f}")


In [40]:
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    classification_report, roc_auc_score
)

probs = model.predict_proba(X_test)[:, 1]

# PR-AUC (Precision-Recall Area Under Curve) is used for imbalanced datasets, as it focuses on the performance
#  of the positive class (flood events) and is more informative than accuracy in such cases.
print(f"PR-AUC: {average_precision_score(y_test, probs):.3f}")

# ROC-AUC (Receiver Operating Characteristic Area Under Curve) is a performance measurement for classification 
# prediction problems at various threshold settings. It tells how much the model is capable of distinguishing
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.3f}")

# don't default to 0.5 — pick a threshold that favors recall (missing a flood is worse)
precision, recall, thresholds = precision_recall_curve(y_test, probs)


PR-AUC: 0.966
ROC-AUC: 0.996


In [41]:
# ── Honest evaluation: is this actually FORECASTING, or just reading the gauge? ──
#
# The two numbers above are computed over every test row, and that flatters the
# model badly. ~82% of the positive labels are rows where the gage is ALREADY
# above action stage, and P(will_flood | already above) = 0.998 — the river does
# not drop back below 5 ft within 24h once it is up. So for most positives,
# "will it flood in the next 24h?" is really "is it flooding right now?", which
# the gage height answers on its own without any model.
#
# The rows that matter operationally are the ones where the river is still BELOW
# action stage. That is the only regime where a forecast has any value, and it is
# where this cell reports the same metrics.
#
# The gage-only baseline is printed next to every number on purpose: it is
# `gage_height_ft` used directly as the score, no model at all. If the model
# cannot clear that baseline by a margin worth having, the 300 trees are not
# earning their keep — and that comparison is invisible unless it is printed
# right here, every time.

from sklearn.metrics import average_precision_score, roc_auc_score

below_stage = (test_df['gage_height_ft'] <= FLOOD_ACTION_STAGE).values
y_true = y_test.values
gage   = test_df['gage_height_ft'].values


def _report(title, mask):
    yy, pp, gg = y_true[mask], probs[mask], gage[mask]
    print(f"\n{title}")
    print(f"  rows {len(yy):>8,}   positives {int(yy.sum()):>7,} ({yy.mean()*100:>5.2f}%)")
    print(f"  {'':<10}{'MODEL':>9}{'gage-only':>11}{'lift':>9}")
    for label, metric in (('PR-AUC', average_precision_score), ('ROC-AUC', roc_auc_score)):
        model_score, baseline_score = metric(yy, pp), metric(yy, gg)
        print(f"  {label:<10}{model_score:>9.3f}{baseline_score:>11.3f}"
              f"{model_score - baseline_score:>+9.3f}")


_report("FULL TEST SET  (the headline numbers above — inflated)",
        np.ones(len(y_true), dtype=bool))
_report("BELOW ACTION STAGE  (the real forecasting task)", below_stage)

print(f"\n{y_true[~below_stage].sum() / y_true.sum() * 100:.1f}% of all positives are rows "
      f"where the gage is ALREADY above {FLOOD_ACTION_STAGE} ft.")



FULL TEST SET  (the headline numbers above — inflated)
  rows  193,313   positives  13,498 ( 6.98%)
                MODEL  gage-only     lift
  PR-AUC        0.966      0.923   +0.043
  ROC-AUC       0.996      0.986   +0.010

BELOW ACTION STAGE  (the real forecasting task)
  rows  182,615   positives   2,824 ( 1.55%)
                MODEL  gage-only     lift
  PR-AUC        0.482      0.171   +0.311
  ROC-AUC       0.982      0.935   +0.047

79.1% of all positives are rows where the gage is ALREADY above 5.0 ft.


In [42]:
def lead_time_last_rise(event_rows, flood_start, threshold=0.5, min_below_hours=6, freq_minutes=15):
    """
    Find the most recent rise above threshold before the flood, requiring
    the probability to have dipped below threshold for at least
    `min_below_hours` beforehand — this separates a fresh, distinct rise
    from an unrelated earlier event or a brief blip.
    """
    trace = event_rows.sort_values('datetime').reset_index(drop=True)
    above = trace['predicted_prob'] >= threshold
    min_below_rows = int(min_below_hours * 60 / freq_minutes)

    crossings = trace.index[above & ~above.shift(1, fill_value=False)]

    valid_crossings = []
    for idx in crossings:
        if idx < min_below_rows:
            continue  # not enough prior history in this window to confirm a real dip — skip, don't assume
        lookback_start = idx - min_below_rows
        if not above.iloc[lookback_start:idx].any():
            valid_crossings.append(idx)

    if not valid_crossings:
        return None  # genuinely no valid crossing found — e.g. sustained risk the whole window

    last_idx = valid_crossings[-1]
    return trace.loc[last_idx, 'datetime']

In [47]:

test_df = test_df.copy()

# Add the predicted probabilities to the test dataframe for further analysis
test_df['predicted_prob'] = probs


# This finds all unique event IDs where the gage height exceeds the flood action stage, indicating a flood event.
flood_events = test_df.loc[test_df['gage_height_ft'] > FLOOD_ACTION_STAGE, 'event_id'].dropna().unique()

results = []

LOOKBACK_HOURS = 168  # how far before the flood to look for an early alert

results = []
for eid in flood_events:
    flood_rows_this_event = test_df[
        (test_df['event_id'] == eid) &
        (test_df['gage_height_ft'] > FLOOD_ACTION_STAGE)
    ]
    flood_start = flood_rows_this_event['datetime'].min() 
    window_start = flood_start - pd.Timedelta(hours=LOOKBACK_HOURS)
    event_rows = test_df[(test_df['datetime'] >= window_start) & (test_df['datetime'] <= flood_rows_this_event['datetime'].max())]
    alert_time = lead_time_last_rise(event_rows, flood_start)
    lead_time = (flood_start - alert_time).total_seconds() / 3600 if alert_time is not None else None

    results.append({'event_id': eid, 'lead_time_hours': lead_time, 'peak_prob': flood_rows_this_event['predicted_prob'].max()})


results_df = pd.DataFrame(results)
print(results_df)
print(f"\nMedian lead time: {results_df['lead_time_hours'].median():.1f}h")

    event_id  lead_time_hours  peak_prob
0      100.0        23.500000   0.999667
1      101.0         6.250000   0.999764
2      102.0        11.500000   0.999641
3      103.0         5.750000   0.999594
4      104.0         9.000000   0.999309
5      105.0        10.250000   0.999845
6      106.0         6.000000   0.999812
7      107.0        19.750000   0.999652
8      108.0        11.250000   0.999571
9      109.0         6.500000   0.999733
10     110.0         4.750000   0.999792
11     111.0         7.000000   0.999535
12     112.0         0.500000   0.999544
13     113.0         3.250000   0.999442
14     114.0         6.250000   0.999459
15     115.0         2.500000   0.999634
16     116.0         7.250000   0.999312
17     117.0        24.000000   0.999730
18     118.0        29.500000   0.999806
19     119.0         5.250000   0.999654
20     120.0        14.000000   0.999715
21     121.0         1.583333   0.998841
22     122.0         5.833333   0.999517
23     123.0    

In [48]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, probs)

# find threshold that gives recall >= some target, e.g. 0.90
import numpy as np
target_recall = 0.90
qualifying = np.where(recall >= target_recall)[0]
idx = qualifying[-1] if len(qualifying) > 0 else -1  # take the LAST (highest-threshold) index, not the first

print(f"Threshold for {target_recall} recall: {thresholds[idx]:.3f}, precision at that point: {precision[idx]:.3f}")

Threshold for 0.9 recall: 0.253, precision at that point: 0.873


In [49]:
from sklearn.metrics import confusion_matrix
final_threshold = thresholds[idx]
preds = (probs >= final_threshold).astype(int)
print(confusion_matrix(y_test, preds))

[[178045   1770]
 [  1349  12149]]


In [50]:
# Generate a classification report
print(classification_report(y_test, preds, target_names=['No Flood', 'Flood']))

              precision    recall  f1-score   support

    No Flood       0.99      0.99      0.99    179815
       Flood       0.87      0.90      0.89     13498

    accuracy                           0.98    193313
   macro avg       0.93      0.95      0.94    193313
weighted avg       0.98      0.98      0.98    193313



In [51]:
# save the model
import joblib

joblib.dump(model, "pot_river_near_little_falls_flood_threshold_xgboost_model.pkl")


['pot_river_near_little_falls_flood_threshold_xgboost_model.pkl']